# gu_toolkit shell overview: slot-based figure layouts

Run the setup cell once, then work top-to-bottom or jump to the shell arrangement you want to inspect.

This notebook is focused on the **Phase 2 shell refactor**:

- declarative shell presets,
- legend placement in several regions,
- shell-level pages that are separate from per-view navigation,
- resize and narrow-host behavior.

Before you start, confirm the notebook is rendering **live widgets** rather than plain text `Figure({...})` reprs. If you only see text reprs, the widget runtime is not active and the interactive checks below will not be meaningful.


In [2]:
!pip -q install plotly pandas scipy numpy sympy anywidget ipywidgets
import import_setup
from gu_toolkit import *
from gu_toolkit._widget_stubs import widgets
from IPython.display import clear_output

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)


## 1) Quick preset map

The shell surface is driven by presets, not by one hard-coded sidebar tree.

Use the table below to orient yourself before opening the live demos:

- `default` is the familiar right-side layout.
- `legend_left` moves only the legend to the left region.
- `legend_bottom` moves only the legend below the stage.
- `legend_hidden` keeps the legend out of the shell entirely.
- `legend_page` moves the legend onto its own shell page.

`legend_right` is accepted as an alias for `default`.


In [3]:
def shell_preset_summary():
    rows = []
    for shell, note in [
        ('default', 'same placement as legend_right'),
        ('legend_left', 'legend moves to left region'),
        ('legend_bottom', 'legend moves below the stage'),
        ('legend_hidden', 'legend omitted from shell presentation'),
        ('legend_page', 'legend gets its own shell page'),
    ]:
        layout = FigureLayout(shell=shell)
        snap = layout.layout_snapshot()
        figure_page = snap['shell_pages']['figure']
        rows.append(
            {
                'shell': shell,
                'note': note,
                'figure.left': ', '.join(figure_page['left_sections']) or '—',
                'figure.right': ', '.join(figure_page['right_sections']) or '—',
                'figure.bottom': ', '.join(figure_page['bottom_sections']) or '—',
                'extra pages': ', '.join(sorted(set(snap['shell_pages']) - {'figure'})) or '—',
                'full-width toggle shown': layout.full_width_checkbox.layout.display != 'none',
            }
        )
    return pd.DataFrame(rows)

shell_preset_summary()

,shell,note,figure.left,figure.right,figure.bottom,extra pages,full-width toggle shown
0,default,same placement as legend_right,—,"legend, parameters, info",—,—,False
1,legend_left,legend moves to left region,legend,"parameters, info",—,—,False
2,legend_bottom,legend moves below the stage,—,"parameters, info",legend,—,False
3,legend_hidden,legend omitted from shell presentation,—,"parameters, info",—,—,False
4,legend_page,legend gets its own shell page,—,"parameters, info",—,legend,False


## 2) Quick contract checks

These checks are deliberately lightweight. They do **not** replace the test suite, but they are useful for confirming that the notebook is running against a Phase 2 build.

What should pass here:

- [ ] default preset keeps legend/parameters/info in the figure page's right region
- [ ] left and bottom presets move only the legend
- [ ] hidden preset removes legend placement at the shell-policy level
- [ ] page preset exposes a second shell page and keeps shell tabs separate from view selection
- [ ] the default title bar does **not** expose the old full-width checkbox


In [4]:
def run_shell_contract_checks():
    rows = []

    def add(name, fn):
        try:
            passed, detail = fn()
        except Exception as exc:
            rows.append({'check': name, 'passed': False, 'detail': f'{type(exc).__name__}: {exc}'})
            return
        rows.append({'check': name, 'passed': bool(passed), 'detail': detail})

    def t_default_right():
        snap = FigureLayout(shell='default').layout_snapshot()
        page = snap['shell_pages']['figure']
        ok = page['right_sections'] == ['legend', 'parameters', 'info']
        return ok, f"right={page['right_sections']}"

    def t_legend_left():
        snap = FigureLayout(shell='legend_left').layout_snapshot()
        page = snap['shell_pages']['figure']
        ok = page['left_sections'] == ['legend'] and page['right_sections'] == ['parameters', 'info']
        return ok, f"left={page['left_sections']}, right={page['right_sections']}"

    def t_legend_bottom():
        snap = FigureLayout(shell='legend_bottom').layout_snapshot()
        page = snap['shell_pages']['figure']
        ok = page['bottom_sections'] == ['legend'] and page['right_sections'] == ['parameters', 'info']
        return ok, f"bottom={page['bottom_sections']}, right={page['right_sections']}"

    def t_legend_hidden():
        snap = FigureLayout(shell='legend_hidden').layout_snapshot()
        page = snap['shell_pages']['figure']
        ok = page['left_sections'] == [] and page['bottom_sections'] == [] and page['right_sections'] == ['parameters', 'info']
        return ok, f"left={page['left_sections']}, bottom={page['bottom_sections']}, right={page['right_sections']}"

    def t_legend_page_and_views_are_separate():
        layout = FigureLayout(shell='legend_page')
        layout.ensure_view_page('main', 'Main')
        layout.ensure_view_page('phase', 'Phase')
        layout.update_sidebar_visibility(has_params=False, has_info=False, has_legend=True)
        snap = layout.layout_snapshot()
        ok = (
            snap['shell_pages']['legend']['main_sections'] == ['legend']
            and snap['visible_shell_page_ids'] == ['figure', 'legend']
            and layout.shell_page_tabs.layout.display == 'flex'
            and layout.view_selector.layout.display == 'flex'
            and layout.view_selector is not layout.shell_page_tabs
        )
        return ok, (
            f"visible_pages={snap['visible_shell_page_ids']}, "
            f"shell_tabs={layout.shell_page_tabs.layout.display}, "
            f"view_selector={layout.view_selector.layout.display}"
        )

    def t_full_width_checkbox_hidden():
        layout = FigureLayout(shell='default')
        ok = layout.full_width_checkbox.layout.display == 'none'
        return ok, f"titlebar full-width display={layout.full_width_checkbox.layout.display}"

    add('default preset keeps secondary sections on the right', t_default_right)
    add('legend_left moves only the legend to the left region', t_legend_left)
    add('legend_bottom moves only the legend below the stage', t_legend_bottom)
    add('legend_hidden omits legend placement', t_legend_hidden)
    add('legend_page creates shell pages distinct from views', t_legend_page_and_views_are_separate)
    add('full-width checkbox is not shown by default', t_full_width_checkbox_hidden)

    return pd.DataFrame(rows)

run_shell_contract_checks()

,check,passed,detail
0,default preset keeps secondary sections on the right,True,"right=['legend', 'parameters', 'info']"
1,legend_left moves only the legend to the left region,True,"left=['legend'], right=['parameters', 'info']"
2,legend_bottom moves only the legend below the stage,True,"bottom=['legend'], right=['parameters', 'info']"
3,legend_hidden omits legend placement,True,"left=[], bottom=[], right=['parameters', 'info']"
4,legend_page creates shell pages distinct from views,True,"visible_pages=['figure', 'legend'], shell_tabs=flex, view_selector=flex"
5,full-width checkbox is not shown by default,True,titlebar full-width display=none


## 3) Helpers for responsive shell demos

The rest of the notebook uses small helper functions to keep each example compact.

Two details matter here:

1. The demo figures are real `Figure(...)` instances using the public `shell=` constructor argument.
2. For resize testing, each figure is mounted into a **framed host box** so you can change the available width without resizing the whole browser window.

That host-box step intentionally reaches into `fig._layout.root_widget` for diagnostics. It is not the everyday authoring path, but it makes the shell regions, shell pages, and reflow behavior much easier to inspect.


In [5]:
def shell_snapshot_frame(fig):
    snap = fig._layout.layout_snapshot()
    rows = []
    for page_id, page in snap['shell_pages'].items():
        rows.append(
            {
                'page': page_id,
                'display': page['display'],
                'includes_stage': page['includes_stage'],
                'main': ', '.join(page['main_sections']) or '—',
                'left': ', '.join(page['left_sections']) or '—',
                'right': ', '.join(page['right_sections']) or '—',
                'bottom': ', '.join(page['bottom_sections']) or '—',
            }
        )
    return pd.DataFrame(rows)


def make_shell_demo(shell='default', *, title=None, with_views=False, include_info=True):
    A, B, C = sp.symbols('A B C')
    fig = Figure(shell=shell, x_range=(-2 * pi, 2 * pi), y_range=(-2.6, 2.6), samples=1200)

    if with_views:
        fig.add_view('phase', title='Phase', x_range=(-2 * pi, 2 * pi), y_range=(-2.6, 2.6), x_label='phase')
        fig.add_view('detail', title='Detail', x_range=(-pi, pi), y_range=(-2.6, 2.6), x_label='detail')

    all_views = ('main', 'phase', 'detail') if with_views else 'main'
    main_views = ('main', 'phase') if with_views else 'main'

    with fig:
        set_title(title or f'Shell preset: {shell}')
        plot(
            A * sin(B * x) + 0.35 * cos((B + 0.5) * x - C),
            x,
            label='signal',
            view=all_views,
            color='royalblue',
        )
        plot(
            0.4 * cos(B * x - C),
            x,
            label='reference',
            view=main_views,
            color='darkorange',
        )
        parameter(A, value=1.2, min=0.2, max=2.0, step=0.1)
        parameter(B, value=1.4, min=0.5, max=3.0, step=0.1)
        parameter(C, value=0.8, min=0.0, max=float(pi), step=0.1)

        if include_info:
            info(
                '<b>What to try</b><br>Change <code>A</code> for amplitude, <code>B</code> for frequency, and <code>C</code> for phase shift.',
                id='guide',
            )

            def diagnostics(fig, ctx):
                p = fig.parameters.snapshot()
                A0, B0, C0 = p['A'], p['B'], p['C']
                return (
                    f'<b>Diagnostics</b><br>'
                    f'render reason: <code>{ctx.reason}</code><br>'
                    f'amplitude ≈ {A0:.2f}<br>'
                    f'period ≈ {2 * float(pi) / B0:.2f}<br>'
                    f'phase shift ≈ {C0:.2f}'
                )

            info(diagnostics, id='diagnostics')

        if with_views:
            info('<b>Main view</b><br>This view shows the shared reference curve.', id='main-note')
            info('<b>Phase view</b><br>Watch the phase-shifted curve appear only here.', id='phase-note', view='phase')
            info('<b>Detail view</b><br>This view keeps the same parameters but zooms the x-range.', id='detail-note', view='detail')

            with fig.views['phase']:
                plot(A * sin(B * (x + C)), x, label='phase shifted', color='purple')

            with fig.views['detail']:
                plot(A * sin(B * x), x, label='detail signal', color='seagreen')

    return fig


def build_shell_lab(shell='default', *, title=None, with_views=False, start_width='1000px'):
    fig = make_shell_demo(shell=shell, title=title, with_views=with_views)

    host = widgets.Box(
        [fig._layout.root_widget],
        layout=widgets.Layout(
            width=start_width,
            max_width='100%',
            border='1px dashed #94a3b8',
            padding='8px',
            overflow='auto',
        ),
    )

    width_control = widgets.ToggleButtons(
        options=['1000px', '820px', '640px', '460px', '340px', '100%'],
        value=start_width,
        description='Host width:',
    )
    reflow_button = widgets.Button(description='Trigger reflow')
    snapshot_button = widgets.Button(description='Show snapshot')
    snapshot_out = widgets.Output(layout=widgets.Layout(width='100%'))

    def refresh_snapshot(*_):
        snap = fig._layout.layout_snapshot()
        with snapshot_out:
            clear_output()
            display(shell_snapshot_frame(fig))
            display(
                pd.DataFrame(
                    [
                        {
                            'shell_preset': snap['shell_preset'],
                            'active_shell_page_id': snap['active_shell_page_id'],
                            'visible_shell_page_ids': ', '.join(snap['visible_shell_page_ids']),
                            'active_view_id': snap['active_view_id'],
                            'view_selector_display': snap['view_selector_display'],
                            'shell_tabs_display': snap['shell_tabs_display'],
                        }
                    ]
                )
            )

    def trigger_reflow(*_):
        snap = fig._layout.layout_snapshot()
        active_view = snap.get('active_view_id')
        if active_view:
            fig._layout.trigger_reflow_for_view(active_view)
        refresh_snapshot()

    def on_width(change):
        if change.get('name') != 'value':
            return
        host.layout.width = change['new']
        trigger_reflow()

    width_control.observe(on_width, names='value')
    reflow_button.on_click(trigger_reflow)
    snapshot_button.on_click(refresh_snapshot)
    refresh_snapshot()

    lab = widgets.VBox(
        [
            widgets.HBox([width_control, reflow_button, snapshot_button]),
            host,
            snapshot_out,
        ]
    )
    return lab, fig


shell_figures = {}


## 4) Default shell: stage with right-side legend, parameters, and info

This is the layout that should feel closest to the pre-refactor notebook shell.

What you should see:

- title at the top,
- the plot stage in the main region,
- legend, parameters, and info stacked on the right,
- output area below,
- **no** full-width checkbox in the title bar.

What to check manually:

- [ ] move the sliders and confirm both the plot and info card update
- [ ] switch the host width to `460px` or `340px` and confirm the plot remains usable instead of being crushed flat
- [ ] resize the whole notebook pane or browser window and confirm the plot continues to fit its active region
- [ ] use **Show snapshot** and confirm the figure page still reports legend/parameters/info on the right


In [6]:
default_lab, shell_figures['default'] = build_shell_lab(
    shell='default',
    title='Default shell preset',
    with_views=False,
    start_width='1000px',
)
default_lab

## 5) Legend on the left

This preset moves the legend to a left-side region while leaving parameters and info on the right.

What you should see:

- legend on the left,
- plot stage in the center,
- parameters and info still on the right.

What to check manually:

- [ ] the legend has moved, but the parameter and info sections have not
- [ ] slider changes still update the same plots and cards
- [ ] at narrower host widths, the figure remains readable and the layout does not collapse into one broken row
- [ ] the snapshot reports `left=legend` and `right=parameters, info`


In [7]:
left_lab, shell_figures['legend_left'] = build_shell_lab(
    shell='legend_left',
    title='Legend on the left',
    with_views=False,
    start_width='1000px',
)
left_lab

## 6) Legend below the stage

This preset keeps parameters and info in the side region, but moves the legend below the stage.

What you should see:

- the plot stage above,
- the legend below the stage,
- parameters and info still on the right.

What to check manually:

- [ ] the legend occupies its own bottom region rather than a side column
- [ ] the bottom legend remains visible and usable at narrow host widths
- [ ] the plot still resizes when you change the host width or notebook width
- [ ] the snapshot reports `bottom=legend`


In [8]:
bottom_lab, shell_figures['legend_bottom'] = build_shell_lab(
    shell='legend_bottom',
    title='Legend below the stage',
    with_views=False,
    start_width='1000px',
)
bottom_lab

## 7) Legend hidden

This preset hides the legend at the shell-policy level while leaving parameters and info visible.

What you should see:

- plot stage,
- parameter section,
- info section,
- **no visible legend section**.

What to check manually:

- [ ] parameters and info still render normally
- [ ] the plot still has multiple labeled traces even though the legend section is not mounted into the shell
- [ ] the figure gains more horizontal room because no legend slot is occupied
- [ ] the snapshot shows no left or bottom legend placement and no extra shell page


In [9]:
hidden_lab, shell_figures['legend_hidden'] = build_shell_lab(
    shell='legend_hidden',
    title='Legend hidden at the shell level',
    with_views=False,
    start_width='1000px',
)
hidden_lab

## 8) Legend on its own shell page

This preset is the first one where **shell-level navigation** becomes visible.

What you should see:

- a shell tab bar with `Figure` and `Legend`,
- the plot stage on the `Figure` page,
- the legend on a separate `Legend` page,
- parameters and info staying on the `Figure` page.

What to check manually:

- [ ] clicking `Legend` hides the stage and shows only the legend page
- [ ] clicking `Figure` brings the stage back
- [ ] shell tabs appear only because this preset actually has more than one available page
- [ ] narrow-host and browser-window resizing still leave the figure page usable when you return to it
- [ ] the snapshot switches `active_shell_page_id` as you click between `Figure` and `Legend`


In [10]:
page_lab, shell_figures['legend_page'] = build_shell_lab(
    shell='legend_page',
    title='Legend on its own shell page',
    with_views=False,
    start_width='1000px',
)
page_lab

## 9) Advanced check: shell pages and views are different navigation layers

This is the most important interaction test in the notebook.

The figure below combines:

- **shell pages** (`Figure` / `Legend`), and
- **view selection** (`main` / `phase` / `detail`).

Those are different concepts and should not collapse into one tab strip.

What to check manually:

- [ ] on the `Figure` shell page, you can switch between `main`, `phase`, and `detail`
- [ ] the `phase` and `detail` views show different plot combinations and info cards
- [ ] sliders continue to drive all views because the parameter state is shared
- [ ] clicking the `Legend` shell page does **not** destroy the last selected view
- [ ] when you come back to the `Figure` shell page, the previously selected view is still active
- [ ] when the `Legend` shell page is active, the stage is not shown, so the view selector should no longer be relevant there
- [ ] the host-width control and **Trigger reflow** still keep the active plot region readable


In [11]:
advanced_lab, shell_figures['legend_page_with_views'] = build_shell_lab(
    shell='legend_page',
    title='Legend page + multiple views',
    with_views=True,
    start_width='820px',
)
advanced_lab

## 10) Current state snapshots

Rerun the next cell **after** you click around in the interactive demos.

It is especially useful after the advanced example above. You should see:

- the active shell page change when you click `Figure` / `Legend`,
- the active view change when you click `main` / `phase` / `detail`,
- shell tabs visible only for the page-based shell presets,
- the view selector visible only for figures that actually have multiple views.


In [12]:
def current_shell_state_table(figures):
    rows = []
    for name, fig in figures.items():
        snap = fig._layout.layout_snapshot()
        rows.append(
            {
                'demo': name,
                'shell_preset': snap['shell_preset'],
                'active_shell_page_id': snap['active_shell_page_id'],
                'visible_shell_page_ids': ', '.join(snap['visible_shell_page_ids']),
                'active_view_id': snap['active_view_id'],
                'view_selector_display': snap['view_selector_display'],
                'shell_tabs_display': snap['shell_tabs_display'],
                'content_layout_mode': snap['content_layout_mode'],
            }
        )
    return pd.DataFrame(rows).sort_values('demo').reset_index(drop=True)

current_shell_state_table(shell_figures)

,demo,shell_preset,active_shell_page_id,visible_shell_page_ids,active_view_id,view_selector_display,shell_tabs_display,content_layout_mode
0,default,default,figure,figure,main,none,none,wrapped
1,legend_bottom,legend_bottom,figure,figure,main,none,none,wrapped
2,legend_hidden,legend_hidden,figure,figure,main,none,none,wrapped
3,legend_left,legend_left,figure,figure,main,none,none,wrapped
4,legend_page,legend_page,figure,"figure, legend",main,none,flex,wrapped
5,legend_page_with_views,legend_page,figure,"figure, legend",main,flex,flex,wrapped


## 11) Manual regression checklist

Use this as a final pass before you close the notebook.

- [ ] the default shell still feels like the old notebook layout, but without the full-width checkbox
- [ ] legend placement can be demonstrated as right/default, left, bottom, hidden, and separate page
- [ ] shell pages are a real separate navigation concept from per-view selection
- [ ] legend/page presets do not require rewriting plot or parameter logic in the notebook examples
- [ ] resizing the host width and the browser window keeps the active plot region usable
- [ ] the snapshot tables agree with what is visible on screen

If something looks wrong, rerun the contract-check cell near the top and compare the live demo snapshots against the preset summary table.
